In [ ]:
# Install required libraries
!pip install transformers datasets peft accelerate




In [ ]:
!pip uninstall -y torchao
!pip install torchao==0.16.0



Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.5 MB/s eta 0:00:00


In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# 1. Load your JSON dataset
# Example JSON file structure:
# [
#   {"Question": "Which bowler took max wickets in innings {innings_number}?",
#    "SQL": "SELECT bowler, COUNT(*) AS total_wickets_per_bowler ..."}
# ]
with open("/content/drive/MyDrive/Automated_cricket_commentary_system/dataset.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)


In [ ]:
# 2. Load base model + tokenizer
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# 3. Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"]  # typical attention modules
)



In [ ]:
# 4. Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
def preprocess_function(examples):
    inputs = [q + "\nSQL:" for q in examples["Question"]]
    targets = examples["SQL"]

    # Pad inputs
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Pad labels
    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Replace pad tokens with -100
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/75 [00:00<?, ? examples/s]

In [ ]:
import transformers
print(transformers.__version__)


5.18.0


In [ ]:
training_args1 = TrainingArguments(
    output_dir="./qwen_sql_lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,   # ✅ stable eval batch
    gradient_accumulation_steps=8,
    learning_rate=5e-5,             # ✅ slightly higher LR
    num_train_epochs=10,            # ✅ fewer epochs
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",          # ✅ correct for Transformers 5.18.0
    eval_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # ✅ keep loss as best metric
    greater_is_better=False,           # ✅ lower loss is better
    warmup_steps=100,
    fp16=True,
)




In [ ]:


training_args = TrainingArguments(
    output_dir="./qwen_sql_lora",
    per_device_train_batch_size=4,     # stable batch size
    per_device_eval_batch_size=4,      # balanced eval batch
    gradient_accumulation_steps=8,     # simulates larger batch
    learning_rate=5e-5,                # slightly higher LR for convergence
    num_train_epochs=10,               # fewer epochs to avoid overfitting
    logging_steps=50,                  # log progress every 50 steps
    logging_strategy="epoch",          # ✅ log training loss each epoch
    save_strategy="epoch",             # save checkpoint each epoch
    eval_strategy="epoch",             # run evaluation each epoch
    eval_steps=100,                    # eval every 100 steps if needed
    save_total_limit=2,                # keep only last 2 checkpoints
    load_best_model_at_end=True,       # reload best checkpoint after training
    metric_for_best_model="eval_loss", # ✅ track loss, no compute_metrics needed
    greater_is_better=False,           # ✅ lower loss is better
    warmup_steps=100,                  # stabilize early training
    fp16=True,                         # mixed precision for memory savings
)


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # causal LM, not masked LM
)




In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset= tokenized_dataset)





In [ ]:
# 8. Train
trainer.train()



Epoch,Training Loss,Validation Loss
1,13.016295,13.194904
2,13.121454,13.183870
3,13.041426,13.165815
4,13.016903,13.150410
5,12.940033,13.117562
6,12.960429,13.075048
7,12.944369,13.029394
8,12.783910,12.972969
9,12.800982,12.912820
10,12.648626,12.847581


TrainOutput(global_step=30, training_loss=12.927442677815755, metrics={'train_runtime': 119.9873, 'train_samples_per_second': 6.251, 'train_steps_per_second': 0.25, 'total_flos': 756011335680000.0, 'train_loss': 12.927442677815755, 'epoch': 10.0})

In [ ]:
# 9. Save LoRA adapter
model.save_pretrained("/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models")
tokenizer.save_pretrained("/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models")

('/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models/tokenizer_config.json',
 '/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models/chat_template.jinja',
 '/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

# Path where you saved the model in Drive
model_path = "/content/drive/MyDrive/Automated_cricket_commentary_system/saved_models"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")





Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Merge LoRA into base weights
model = PeftModel.from_pretrained(base_model, model_path)
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("/content/drive/MyDrive/Automated_cricket_commentary_system/merged_models")
tokenizer.save_pretrained("/content/drive/MyDrive/Automated_cricket_commentary_system/merged_models")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Automated_cricket_commentary_system/merged_models/tokenizer_config.json',
 '/content/drive/MyDrive/Automated_cricket_commentary_system/merged_models/chat_template.jinja',
 '/content/drive/MyDrive/Automated_cricket_commentary_system/merged_models/tokenizer.json')

In [ ]:
instruction = """
Convert the following natural language query into a valid SQL statement.

Guidelines:
- Output only the SQL query, nothing else.
- Always use parentheses around OR conditions when combined with AND.
- Do not add WHERE conditions for year, stage, or match_id unless explicitly mentioned.
- Do not add WHERE clauses unless the query explicitly mentions innings, player name, or other filters.
- Do not add a WHERE clause for innings unless the query explicitly mentions an innings number (e.g., "in the second innings").
- If no innings number is mentioned, do not include any innings filter.
- Never add a WHERE clause on batter unless the query explicitly mentions a particular player name.
- Do not invent filters on columns like actual_delivery, stage, match_id, or year.
- Use aggregation (SUM, MAX, COUNT, MIN, AVG) when the query implies totals, highest, lowest, or "so far".
- Use GROUP BY when aggregating per player, team, or innings.
- Always use column names exactly as they appear in the provided schema.
- Do not invent new column names or misspell existing ones.
- If a column name is uncertain, default to the schema definition rather than guessing.
- Do not group by columns other than batter, team, or innings unless explicitly mentioned in the query.
- Do not add WHERE batter = '' or any empty filter.
- For "Who..." queries, always aggregate by batter only.
- Always alias aggregate outputs with a consistent name: use AS total_runs for batter queries, and AS total_runs_team for team queries.
- If a column name is uncertain, consult the provided schema and use only those columns. Do not guess or invent.
- Do not add conditions like batter!='' or team!='' unless explicitly mentioned in the query.




Rules for runs/score queries:
- If the query begins with "Which team", always use team totals (including extras) with SUM(runs_total), grouped by team.
- Otherwise, use individual batter totals with SUM(runs_batter), grouped by batter.
- If the query asks "Who achieved the highest score?", "Who achieved the highest runs?", "Which player scored the highest?", or similar → use individual batter totals with SUM(runs_batter), grouped by batter.
- Treat "score" and "runs" as synonyms.
- Treat "achieved", "got", "scored", "secured", "made", "hit", "knocked", "registered" as synonyms in the context of "Who..." or "Which player..." or "Which team...".
- Always include the batter name in SELECT when the query is about "Who" or "Which player".
- Always include the team name in SELECT when the query is about "Which team".
- Always format aggregate functions in uppercase without spaces: e.g., SUM(runs_batter), COUNT(*), MAX(value).
- Always alias aggregate outputs with a clear name (e.g., AS total_runs, AS total_score).

Final requirements:
- Always end SQL with a semicolon.
- Follow instructions strictly. No hallucination.
"""


In [ ]:
# Schema
schema = """
Database schema:
Table: deliveries
Columns:
id bigint UN AI PK,
year int,
stage varchar(50),
match_id int,
innings int,
team varchar(100),
over_number int,
ball_number int,
actual_delivery varchar(20),
bowler varchar(100),
batter varchar(100),
non_striker varchar(100),
runs_batter int,
runs_extras int,
runs_total int,
extras_type varchar(50),
boundary_4 tinyint(1),
boundary_6 tinyint(1),
wicket tinyint(1),
wicket_kind varchar(50),
player_out varchar(100),
fielder varchar(100),
score_after_ball varchar(20)
"""



In [ ]:
import re

# --- Step 6: Generate SQL directly ---
def generate_sql(schema, instruction, query):


    prompt = f"""
    Database Schema:
    {schema}

    General Instruction:
    {instruction}



    User Query:
    {query}

    SQL query:

    """

    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate only continuation (skip prompt echo)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        return_dict_in_generate=True
    )

    # Decode only the new tokens after the prompt
    raw_sql = tokenizer.decode(
        outputs.sequences[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # --- Extraction: keep everything from SELECT until ; ---
    match = re.search(r"(SELECT.*?;)", raw_sql, re.DOTALL | re.IGNORECASE)
    if match:
        sql_query = match.group(1).strip()
    else:
        sql_query = raw_sql

    # --- Post-processing fixers ---
    def fix_sql_parentheses(sql: str) -> str:
        if "AND" in sql and "OR" in sql:
            sql = re.sub(r"AND (.+) OR (.+)", r"AND (\1 OR \2)", sql)
        return sql

    def fix_sql_semicolon(sql: str) -> str:
        sql = re.sub(r";\)", ")", sql)   # fix misplaced semicolon inside parentheses
        sql = sql.rstrip()
        if not sql.endswith(";"):
            sql += ";"
        return sql

    sql_query = fix_sql_parentheses(sql_query)
    sql_query = fix_sql_semicolon(sql_query)

    print("Natural language query:", query)
    print(" Generated SQL query:", sql_query)

    return sql_query


In [ ]:
import re

def normalize_sql(sql_query: str) -> str:
    # Regex to catch aggregate functions with optional spaces
    sql_query = re.sub(r"\bsum\s*\(", "SUM(", sql_query, flags=re.IGNORECASE)
    sql_query = re.sub(r"\bcount\s*\(", "COUNT(", sql_query, flags=re.IGNORECASE)
    sql_query = re.sub(r"\bmax\s*\(", "MAX(", sql_query, flags=re.IGNORECASE)
    sql_query = re.sub(r"\bmin\s*\(", "MIN(", sql_query, flags=re.IGNORECASE)
    sql_query = re.sub(r"\bavg\s*\(", "AVG(", sql_query, flags=re.IGNORECASE)
    return sql_query


In [ ]:
import re

def clean_sql(nl_query: str, sql_query: str) -> str:
    q = nl_query.lower()

    # Remove phantom team filters
    if "team" not in q:
        sql_query = re.sub(r"where\s+team\s*=\s*['\"].*?['\"]", "", sql_query, flags=re.IGNORECASE)

    # Remove string innings filters like '1st Innings'
    sql_query = re.sub(r"where\s+innings\s*=\s*['\"].*?['\"]", "", sql_query, flags=re.IGNORECASE)

    # Ensure grouping is correct for "Who..." queries
    if "who" in q or "which player" in q:
        sql_query = re.sub(r"group by.*", "GROUP BY batter", sql_query, flags=re.IGNORECASE)

    return sql_query.strip()


In [ ]:
nl_query="Who achieved the highest score ?"
sql_query = generate_sql(schema, instruction, nl_query)
sql_query = normalize_sql(sql_query)
sql_query =clean_sql(nl_query, sql_query)


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:2944: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Natural language query: Who achieved the highest score ?
 Generated SQL query: SELECT 
        CASE 
            WHEN T2.team IS NOT NULL THEN T2.total_runs_team 
            ELSE T1.total_runs 
        END AS total_runs 
     FROM (
         SELECT 
             T1.batter AS batter, 
             SUM(T1.runs_batter) AS total_runs 
         FROM 
             deliveries AS T1 
         WHERE 
             T1.innings BETWEEN 1 AND 3 
         GROUP BY 
             T1.batter 
     ) AS T1 
     LEFT JOIN 
         (SELECT * FROM deliveries WHERE innings = 3) AS T2 ON T1.batter = T2;


In [ ]:
sql_query

'SELECT \n        CASE \n            WHEN T2.team IS NOT NULL THEN T2.total_runs_team \n            ELSE T1.total_runs \n        END AS total_runs \n     FROM (\n         SELECT \n             T1.batter AS batter, \n             SUM(T1.runs_batter) AS total_runs \n         FROM \n             deliveries AS T1 \n         WHERE \n             T1.innings BETWEEN 1 AND 3 \n         GROUP BY batter\n             T1.batter \n     ) AS T1 \n     LEFT JOIN \n         (SELECT * FROM deliveries WHERE innings = 3) AS T2 ON T1.batter = T2;'